# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

# Simple Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))
print(chain.invoke(input= {'city': '강원도'}))

# Sequential Chain

In [ ]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm 

eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""


print(chain1.invoke(eng_text))


chain2 = prompt2 | llm | output_parser
kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""
print(chain2.invoke(kor_text))

In [ ]:
chain = chain1 | chain2

print(chain.invoke({'eng_text': eng_text}))

# Conditional Chain

In [ ]:
from langchain_core.runnables import RunnableBranch

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단게적인 풀이를 수식(LaTex)와 함께 작성해주세요. {question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 친절하고, 감석적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')
    return '계산' in question or 'calc' in question

branch_chain = RunnableBranch(
    (is_math_question, math_chain),
    default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산해줘.'}))

### Memory Chain

'RunnableWithMessageHistory'를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory              # LangChain 대화기록 메모리 저장용 클래스
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage    # 메시지 타입들
from pydantic import BaseModel, Field       # Pydantic 모델(검증/ 기본값 생성) 도구
from typing import List     # 타입 힌트(List)

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    # Field(default_factory = list) : 인스턴스마다 독립적인 messages list 를 구성
    messages : List[BaseMessage] = Field(default_factory= list)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)      # 전달받은 메시지들을 기존 리스트 뒤에 추가
    
    def clear(self) -> None:
        self.messages = []  # 저장된 메시지들을 초기화
        
store = {}  # {session_id : 히스토리 객체(InMemoryHistory)} 저장소

def get_by_session_id(session_id: str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()   # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]        # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1')   # 세션 ID '1'의 히스토리 가져오기(없으면 메모리 공간 생성)
history1.add_messages([AIMessage(content= '반갑습니다. Capybara님!')])      # AI 메시지 추가
history1.add_messages([HumanMessage(content= '그래~ 나 Cap이야~ 만나서 반갑다!!')]) # 유저 메시지 추가
print(f'{history1 = }')     


history2 = get_by_session_id('2')
print(f'{history2 = }')


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name= 'history'),
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key= 'question',
    history_messages_key= 'history'
)

chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지를 3마리 키우고 있습니다.'
}, config = {
    'configurable': {
        'session_id': '100'
    }
}
)